# Separated xNES: joint tuning and frozen CMA-ES comparison

Mean and traceless shape use directional evidence; scale uses unsigned movement power. Each has an independent trust limiter. This is rate adaptation, not momentum. A signal threshold of 2 means twice the conditional random-ranking power, not an 80% confidence level or a literal SNR of two.

Default ceiling: **13,094,400 optimizer observations**, versus slim's 13,440,000. Clean diagnostic assessments are extra, as in the benchmark. Three fresh training panels promote 24 → 8 → 3 configurations on instances 1, 2, 3. Freeze the winner before validation on instances 4–5 and all five slim dimensions. These instances are held out from this search, not earlier repository experiments. This is evidence within BBOB, not universal superiority.

The score averages log improvement of the clean recommendation at 10%, 33%, and 100% of budget. Numerical failures receive −8 rather than disappearing from averages. Early stops carry the last mean forward; unused observations are not recycled. Short-budget screening can eliminate late-blooming settings.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'leitwerk').is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from leitwerk import XNESLearningRates
from benchmarks.runner import PRESETS
from benchmarks.tuning import TuningPlan, candidates, load_panel, run_tuning


## Meaningful knobs

| Parameter | Meaning | Search interval |
|---|---|---|
| `eta_mean` | Initial mean rate | 0.3–1 |
| `eta_scale_global` | Initial log-covariance scale rate | 0.15–1 |
| `eta_scale_shape` | Multiplier of canonical dimension-dependent shape rate | 0.08–1.5 |
| `signal_threshold` | Power/reference threshold for rate growth | 1.2–4 |
| `evidence_half_life` | Mean/shape amplitude half-life in generations; scale power halves twice as fast | 8–64 |
| `rate_half_life` | Fastest rate-halving time in generations | 4–32 |
| `scale_recovery` | Scale-rate growth is this many times slower than shrinkage | 1–8 |
| `max_kl` | Isolated KL ceiling per block, **not total Gaussian KL** | 0.05–2 |

Defaults and their fixed-rate ablation enter first; other proposals jointly cover log-scaled ranges with a seeded Latin hypercube. Initial rates still matter at short budgets. Adaptation has rate ceilings and a 0.001-of-ceiling floor, outside this search. A fixed-rate winner is a legitimate result against the adaptation hypothesis.

In [ ]:
PLAN = TuningPlan()
LOAD_RUN = None  # Existing tuning-run path to analyze without spending observations.
budget = pd.DataFrame(PLAN.budget())
display(budget)
print(f'Total: {budget.observations.sum():,}; slim: {PRESETS["slim"].runs * PRESETS["slim"].budget:,}')
display(pd.DataFrame({k: asdict(v) for k, v in candidates(PLAN.counts[0], PLAN.seed).items()}).T)


In [ ]:
directory = Path(LOAD_RUN) if LOAD_RUN is not None else run_tuning(PLAN)
print(directory)
assert (directory / 'complete.json').exists(), 'Incomplete run: inspect artifacts before comparing validation.'
saved_plan = json.loads((directory / 'tuning-plan.json').read_text())
selected = json.loads((directory / 'selected.json').read_text())
display(selected)
recommended_rates = XNESLearningRates(**selected['learning_rates'])
# Use with XNES.update or Optimizer(learning_rates=...). Core defaults are never rewritten.


In [ ]:
for stage in range(len(saved_plan['plan']['stages'])):
    frame = pd.DataFrame(load_panel(directory, f'search-{stage}'))
    print(f'Stage {stage}: scores are not comparable across fresh panels')
    display(frame.groupby('variant').agg(score=('score', 'mean'), failures=('failed', 'sum'), observations=('evaluations', 'sum')).sort_values('score', ascending=False))


## Held-out comparison

Compare the frozen selection, untouched defaults, selection with adaptation disabled (same trust limits), and CMA-ES. Noise regimes remain separate: an aggregate win can hide a clean regression. Scores and final-gap wins answer different questions. Failures lose against successful runs, two failures tie, and final gaps tie below the scoring floor. Two held-out instances give limited statistical precision; do not retune on this panel.

In [ ]:
heldout = pd.DataFrame(load_panel(directory, 'held-out'))
display(heldout.groupby(['noise', 'variant']).agg(score=('score', 'mean'), failures=('failed', 'sum'), observations=('evaluations', 'sum'), seconds=('seconds', 'sum')))
keys = ['function', 'dimension', 'instance', 'repeat', 'noise']
paired = heldout.pivot(index=keys, columns='variant', values=['score', 'relative_gap', 'failed'])
comparisons = []
for variant in ['selected', 'default', 'fixed']:
    a = np.log10(np.maximum(paired['relative_gap'][variant].astype(float), 1e-8))
    b = np.log10(np.maximum(paired['relative_gap']['cma'].astype(float), 1e-8))
    fa, fb = paired['failed'][variant].astype(bool), paired['failed']['cma'].astype(bool)
    tie = (fa & fb) | (~fa & ~fb & np.isclose(a, b, atol=1e-8, rtol=0))
    win = ~fa & (fb | ((a < b) & ~tie))
    table = pd.DataFrame({'win': win, 'tie': tie, 'score_delta': paired['score'][variant] - paired['score']['cma']})
    summary = table.groupby(level='noise').mean()
    summary['variant'] = variant
    comparisons.append(summary)
display(pd.concat(comparisons))
delta = (paired['score']['selected'] - paired['score']['cma']).astype(float).reset_index(name='advantage')
fig, axes = plt.subplots(1, len(delta.noise.unique()), figsize=(17, 5), constrained_layout=True, squeeze=False)
for ax, (noise, group) in zip(axes.flat, delta.groupby('noise')):
    grid = group.pivot_table(index='function', columns='dimension', values='advantage')
    mesh = ax.imshow(grid, aspect='auto', cmap='RdBu', vmin=-2, vmax=2)
    ax.set(title=noise, xlabel='dimension', ylabel='BBOB function', xticks=range(len(grid.columns)), xticklabels=grid.columns, yticks=range(len(grid.index)), yticklabels=grid.index)
fig.colorbar(mesh, ax=list(axes.flat), label='selected − CMA score (positive favors xNES)')
plt.show()


In [ ]:
diagnostics = pd.json_normalize(heldout.diagnostics).add_prefix('final_')
clipping = pd.json_normalize(heldout.clipping).add_prefix('clipped_fraction_')
health = pd.concat([heldout[['variant', 'noise']].reset_index(drop=True), diagnostics, clipping], axis=1)
columns = [c for c in health if c.startswith(('final_rate_', 'final_signal_', 'clipped_fraction_'))]
display(health.groupby(['noise', 'variant'])[columns].median())
print('Recorded optimizer observations:', heldout.evaluations.sum(), '(validation only)')
print('Clipping fractions weight generations, not evaluations. Persistent clipping suggests oversized proposed rates.')
print('Signals are power ratios, not probabilities. Do not retune on the held-out panel.')
